In [ ]:
import numpy as np
import cv2
import json
import os
import random
from scipy.interpolate import splprep, splev

IMG_SIZE            = (512, 512)
NUM_IMAGES_PER_TIER = 10
OUTPUT_DIR          = "synthetic_fault_lines_realistic"

# Tiers focus on geometry only — all have the same background noise
TIERS = {
    "group1_straight":      dict(curvature=False, gaps=False, thick=(1,2),  n_lines=(2,6),   width_range=(2,4)),
    "group2_gaps":          dict(curvature=False, gaps=True,  thick=(3,6),  n_lines=(2,6),   width_range=(2,4)),
    "group3_curved":        dict(curvature=True,  gaps=False, thick=(3,6),  n_lines=(4,10),  width_range=(2,4)),
    "group4_dense_overlap": dict(curvature=True,  gaps=True,  thick=(2,8),  n_lines=(10,20), width_range=(2,6)),
    "group5_mixed_widths":  dict(curvature=False, gaps=False, thick=(1,16), n_lines=(2,6),   width_range=(1,16)),
}

NOISE_STD = 25   # fixed background noise for all tiers

os.makedirs(OUTPUT_DIR, exist_ok=True)
for tier in TIERS:
    os.makedirs(os.path.join(OUTPUT_DIR, tier), exist_ok=True)



# ── Geometry ──────────────────────────────────────────────────────────────────

def random_spline_points(n_points=8, curvature=0.4):
    x   = np.linspace(40, IMG_SIZE[1]-40, n_points) + np.random.uniform(-10, 10, n_points)
    y   = np.linspace(80, IMG_SIZE[0]-80, n_points) + np.random.normal(0, 5 + 20*curvature, n_points)
    pts = np.column_stack((x, y))
    angle = np.random.uniform(-np.pi/6, np.pi/6)
    rot   = np.array([[np.cos(angle), -np.sin(angle)],
                      [np.sin(angle),  np.cos(angle)]])
    pts   = (pts @ rot.T)
    pts   = np.clip(pts, [0,0], [IMG_SIZE[1]-1, IMG_SIZE[0]-1])
    pts   = pts[np.argsort(pts[:,0])]
    if len(pts) < 6:
        return pts.astype(np.int32)
    try:
        tck, _ = splprep([pts[:,0], pts[:,1]], s=curvature*30, k=1)
        x_new, y_new = splev(np.linspace(0,1,120), tck)
        return np.column_stack((x_new, y_new)).astype(np.int32)
    except Exception:
        return pts.astype(np.int32)


def generate_piecewise_linear(n_segments=2):
    pts       = []
    x         = random.uniform(80, IMG_SIZE[1]-80)
    y         = random.uniform(80, IMG_SIZE[0]-80)
    direction = random.uniform(-np.pi, np.pi)
    for _ in range(n_segments):
        length     = random.uniform(150, 420)
        direction += np.random.normal(0, 0.08)
        pts.append([x, y])
        x = np.clip(x + length*np.cos(direction), 30, IMG_SIZE[1]-30)
        y = np.clip(y + length*np.sin(direction), 30, IMG_SIZE[0]-30)
    return np.array(pts, dtype=np.int32)


def generate_fault_network(n_lines_range, high_curvature):
    lines = []
    for _ in range(random.randint(*n_lines_range)):
        if random.random() < (0.6 if high_curvature else 0.9):
            pts = generate_piecewise_linear(n_segments=random.randint(1,3))
        else:
            pts = random_spline_points(curvature=random.uniform(0.3,0.7))
        if len(pts) >= 20:
            lines.append(pts)
    return lines


def add_gaps(lines, gap_prob=0.3):
    gapped = []
    for pts in lines:
        if random.random() > gap_prob or len(pts) < 20:
            gapped.append(pts)
            continue
        n_gaps  = random.randint(1, 2)
        mask    = np.ones(len(pts), dtype=bool)
        for _ in range(n_gaps):
            start   = random.randint(0, len(pts)-10)
            gap_len = random.randint(5, max(6, len(pts)//4))
            mask[start:start+gap_len] = False
        segments = []
        seg = []
        for k, keep in enumerate(mask):
            if keep:
                seg.append(pts[k])
            elif seg:
                if len(seg) >= 10:
                    segments.append(np.array(seg, dtype=np.int32))
                seg = []
        if len(seg) >= 10:
            segments.append(np.array(seg, dtype=np.int32))
        gapped.extend(segments)
    return gapped


# ── Background ────────────────────────────────────────────────────────────────

def make_background():
    """Perlin-like background: smooth low-frequency texture + Gaussian noise."""
    # Low-frequency texture via blurred noise
    base  = np.random.randint(80, 160, IMG_SIZE, dtype=np.uint8)
    bg    = cv2.GaussianBlur(base, (61, 61), 20).astype(np.float32)
    noise = np.random.normal(0, NOISE_STD, IMG_SIZE)
    return np.clip(bg + noise, 0, 255).astype(np.uint8)


# ── Rasterise ─────────────────────────────────────────────────────────────────

def rasterize_lines(canvas, lines, thickness_range):
    for pts in lines:
        thick = random.randint(*thickness_range)
        for i in range(len(pts)-1):
            cv2.line(canvas, tuple(pts[i]), tuple(pts[i+1]),
                     255, thick, lineType=cv2.LINE_8)


# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    for tier, cfg in TIERS.items():
        print(f"Generating {NUM_IMAGES_PER_TIER} images for {tier} ...")
        tier_dir = os.path.join(OUTPUT_DIR, tier)

        for i in range(NUM_IMAGES_PER_TIER):

            for attempt in range(20):
                lines = generate_fault_network(
                    n_lines_range=cfg["n_lines"],
                    high_curvature=cfg["curvature"],
                )
                if cfg["gaps"]:
                    lines = add_gaps(lines)

                fault_canvas = np.zeros(IMG_SIZE, dtype=np.uint8)
                rasterize_lines(fault_canvas, lines, cfg["thick"])

                if fault_canvas.max() > 0:
                    break
            else:
                print(f"  WARNING: skipping {tier} img {i} — empty after 20 attempts")
                continue

            # Composite: dark faults on textured background
            bg    = make_background()
            alpha = fault_canvas.astype(np.float32) / 255.0
            final = np.clip(
                bg.astype(np.float32) * (1 - 0.6 * alpha), 0, 255
            ).astype(np.uint8)

            img_path = os.path.join(tier_dir, f"img_{i:04d}.png")
            gt_path  = os.path.join(tier_dir, f"img_{i:04d}_gt.json")

            cv2.imwrite(img_path, final)

            gt = [{"points": pts.tolist()} for pts in lines]
            with open(gt_path, "w") as f:
                json.dump({"lines": gt, "image_size": list(IMG_SIZE)}, f, indent=2)

            if (i+1) % 20 == 0:
                print(f"  {i+1}/{NUM_IMAGES_PER_TIER}")

    print(f"\nDone. Saved to {OUTPUT_DIR}/")


if __name__ == "__main__":
    random.seed(42)
    np.random.seed(42)
    main()


Generating 10 images for group1_straight ...
Generating 10 images for group2_gaps ...
Generating 10 images for group3_curved ...
Generating 10 images for group4_dense_overlap ...
Generating 10 images for group5_mixed_widths ...

Done. Saved to synthetic_fault_lines_realistic/
